[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/07_batchnorm.ipynb)

# 🟡 Medium: Implement BatchNorm

Implement **Batch Normalization** with both **training** and **inference** behavior.

In training mode, use **batch statistics** and update running estimates:

$$\text{BN}(x) = \gamma \cdot \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}} + \beta$$

where $\mu_B$ and $\sigma_B^2$ are the mean and variance computed **across the batch** (dim=0).

In inference mode, use the provided **running mean/var** instead of current batch stats.

### Signature
```python
def my_batch_norm(
    x: torch.Tensor,
    gamma: torch.Tensor,
    beta: torch.Tensor,
    running_mean: torch.Tensor,
    running_var: torch.Tensor,
    eps: float = 1e-5,
    momentum: float = 0.1,
    training: bool = True,
) -> torch.Tensor:
    # x: (N, D) — normalize each feature across all samples in the batch
    # running_mean, running_var: updated in-place during training; used as-is during inference
```

### Rules
- Do **NOT** use `F.batch_norm`, `nn.BatchNorm1d`, etc.
- Compute batch mean and variance over `dim=0` with `unbiased=False`
- Update running stats like PyTorch: `running = (1 - momentum) * running + momentum * batch_stat`
- Use `running_mean` / `running_var` for inference when `training=False`
- Must support autograd w.r.t. `x`, `gamma`, `beta`（running statistics 应视作 buffer，而不是需要梯度的参数）

In [1]:
import torch

In [12]:
# ✏️ YOUR IMPLEMENTATION HERE
    # if training:
    #     batch_mean = x.mean(dim=0)
    #     batch_var = x.var(dim=0, unbiased=False)

    #     # Update running statistics in-place. Detach to avoid tracking gradients.
    #     running_mean.mul_(1 - momentum).add_(momentum * batch_mean.detach())
    #     running_var.mul_(1 - momentum).add_(momentum * batch_var.detach())

    #     mean = batch_mean
    #     var = batch_var
    # else:
    #     mean = running_mean
    #     var = running_var

    # x_norm = (x - mean) / torch.sqrt(var + eps)
    # return gamma * x_norm + beta

def my_batch_norm(
    x,
    gamma,
    beta,
    running_mean,
    running_var,
    eps=1e-5,
    momentum=0.1,
    training=True,
):
    mu = torch.mean(x,dim=0)
    var = torch.var(x,dim=0,unbiased=False)

    running_mean = (1-momentum)*running_mean + momentum*mu.detach()
    running_var = (1-momentum)*running_var + momentum*var.detach()
    
    if not training:
        mu = running_mean
        var = running_var
        
    return gamma*(x-mu)/torch.sqrt(var+eps)+beta


In [15]:
# 🧪 Debug
x = torch.randn(8, 4)
gamma = torch.ones(4)
beta = torch.zeros(4)

# Running stats typically live on the same device and shape as features
running_mean = torch.zeros(4)
running_var = torch.ones(4)

# Training mode: uses batch stats and updates running_mean / running_var
out_train = my_batch_norm(x, gamma, beta, running_mean, running_var, training=True)
print("[Train] Output shape:", out_train.shape)
print("[Train] Column means:", out_train.mean(dim=0))   # should be ~0
print("[Train] Column stds: ", out_train.std(dim=0))    # should be ~1
print("Updated running_mean:", running_mean)
print("Updated running_var:", running_var)

# Inference mode: uses running_mean / running_var only
out_eval = my_batch_norm(x, gamma, beta, running_mean, running_var, training=False)
print("[Eval] Output shape:", out_eval.shape)

[Train] Output shape: torch.Size([8, 4])
[Train] Column means: tensor([-2.9802e-08,  0.0000e+00,  7.4506e-09,  1.4901e-08])
[Train] Column stds:  tensor([1.0690, 1.0690, 1.0690, 1.0690])
Updated running_mean: tensor([ 0.0642,  0.0594,  0.0314, -0.0385])
Updated running_var: tensor([0.9980, 0.9470, 1.0242, 0.9841])
[Eval] Output shape: torch.Size([8, 4])


In [ ]:
# ✅ SUBMIT
from torch_judge import check
check("batchnorm")

In [18]:
from torch_judge import status
status()


────────────────────────────────────────────────────────
  🔥 TorchCode Progress: 14/40 solved
────────────────────────────────────────────────────────
  ✅ cross_entropy        [Easy]  ⚡ 10.7ms  (3 attempts)
     Cross-Entropy Loss
  ✅ dropout              [Easy]  ⚡ 12.5ms  (2 attempts)
     Implement Dropout
  ✅ embedding            [Easy]  ⚡ 7.5ms  (3 attempts)
     Embedding Layer
  ✅ gelu                 [Easy]  ⚡ 14.3ms  (1 attempts)
     GELU Activation
  ✅ gradient_accumulation [Easy]  ⚡ 19.2ms  (2 attempts)
     Gradient Accumulation
  ✅ gradient_clipping    [Easy]  ⚡ 11.7ms  (1 attempts)
     Gradient Norm Clipping
  ✅ relu                 [Easy]  ⚡ 91.5ms  (1 attempts)
     Implement ReLU
  ✅ softmax              [Easy]  ⚡ 8.1ms  (1 attempts)
     Implement Softmax
  ✅ weight_init          [Easy]  ⚡ 18.6ms  (3 attempts)
     Kaiming Initialization
  ⏳ adam                 [Medium]
     Adam Optimizer
  ✅ batchnorm            [Medium]  ⚡ 7.8ms  (5 attempts)
     Implement Batc